# Following vectors and constructing generated submodules

A module can have a nonzero space at a parameter even though the particular vector we started with has already died. Here we follow chosen coordinates through the actual maps, then construct the submodule they generate and the corresponding quotient.

We start from two square-supported modules on a finite grid. This is a direct algebraic input: no point cloud or filtration is needed. The encoding also retains a classifier so that we can view the finite answer on parameter regions. You need Julia, TamerOp and CairoMakie; familiarity with vectors and matrix multiplication is enough. See the [spaces and maps guide](../spaces_and_maps.md) for the surrounding workflow.


In [ ]:
import TamerOp as OP
using CairoMakie
A = OP.Advanced


## Two squares and their overlap

The first square is $[0,2)\times[1,3)$; the second is $[1,3)\times[0,2)$. We give each active square one coordinate. In the overlap the stalk is two-dimensional, with basis $e_1,e_2$. Across a cover relation the map preserves a coordinate exactly when its square is active at both ends.

The finite grid uses labels with the first coordinate varying fastest. Label 6 represents the overlap cell $[1,2)\times[1,2)$. The classifier is defined on the nonnegative quadrant; points with a negative coordinate are unrepresented, which is different from a represented zero stalk.


In [ ]:
P = A.ProductOfChainsPoset((4, 4))
points = [(x, y) for y in 0:3 for x in 0:3]
active = [findall(identity, [0 <= x < 2 && 1 <= y < 3,
                           1 <= x < 3 && 0 <= y < 2]) for (x, y) in points]
dims = length.(active)
maps = Dict((u, v) => OP.QQ[a == b for a in active[v], b in active[u]]
            for (u, v) in A.cover_edges(P))
M = A.PModule{OP.QQ}(P, dims, maps; field=OP.CoreModules.QQField())
classifier = A.GridEncodingMap(P, (collect(0:3), collect(0:3)))
enc = OP.EncodingResult(P, M, classifier)
OP.describe(enc)


## Follow one vector

`source` is a **finite label**. `vectors` contains coordinates in that stalk's stored basis. The result retains the images and their ranks; it does not construct a new module or a quotient yet.


In [ ]:
first_vector = OP.propagate(enc; source=6, vectors=[1, 0])
@assert OP.propagation_summary(first_vector).nonzero_support == [6, 10]
OP.propagation_summary(first_vector)


In [ ]:
OP.visualize(first_vector)


The upper part of the first square retains $e_1$. In the right-hand part of the other square its image is zero even though the ambient stalk is nonzero. At labels outside the source upper ideal there is no forward map at all.

Choose label 7, the cell immediately to the right, to see the matrix that kills $e_1$.


In [ ]:
OP.visualize(first_vector; target=7)


## Change the chosen vector

The second basis vector survives in the other direction. Their sum survives in either direction: it is a single vector, so its image has dimension at most one even where the original module is two-dimensional.


In [ ]:
second_vector = OP.propagate(enc; source=6, vectors=[0, 1])
sum_vector = OP.propagate(enc; source=6, vectors=[1, 1])
@assert OP.propagation_summary(second_vector).nonzero_support == [6, 7]
@assert OP.propagation_summary(sum_vector).nonzero_support == [6, 7, 10]
OP.visualize(sum_vector; target=10)


These supports are pullbacks of the finite-label answer. We have not selected an original source point inside its fiber. In particular, coloring an entire fiber does not assert that every point in it lies above an arbitrary point one might choose later.

For a subspace, put spanning vectors in the columns. Dependent columns are allowed and remain visible; the reported image dimension is their rank.


In [ ]:
plane = OP.propagate(enc; source=6, vectors=[1 0; 0 1])
@assert OP.dimensions(plane)[6] == 2
@assert OP.dimensions(plane)[7] == 1
OP.visualize(plane; size=(1600, 770))


## Recover the generated submodule

The generated submodule has the chosen image at every vertex above the source, and a zero stalk elsewhere. Its inclusion remembers how its own basis sits inside the ambient module. This extra construction supplies structure maps, so it is useful when we want to do more algebra with the selection.


In [ ]:
S = OP.generated_submodule(sum_vector)
N = A.submodule_object(S)
inclusion = A.inclusion_map(S)
@assert OP.dimensions(N).stalks == [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
@assert A.check_submodule(S).valid
OP.describe(S)


The inclusion commutes with each structure map. We can inspect that equation at labels 6 and 7 using the ordinary morphism viewer. Both routes in the square send the generated source coordinate to the same ambient vector. Here the viewer’s M is the generated module and its N is the ambient module.


In [ ]:
OP.visualize(inclusion; kind=:naturality, pair=(6, 7))


The quotient removes the generated submodule. Its projection annihilates every included vector. A zero-dimensional quotient or an empty matrix is a legitimate result.


In [ ]:
quotient, projection = A.quotient_with_projection(S)
@assert all(iszero(A.component(projection, q) * A.component(inclusion, q)) for q in 1:16)
OP.visualize(projection; vertex=6)


## A submodule need not split

Consider the identity map between two one-dimensional spaces on a chain. A vector born at the second vertex generates the submodule with dimensions $(0,1)$. A complementary submodule would need dimensions $(1,0)$, but the identity sends its first nonzero space into the second. Such a complement cannot be closed under the maps.


In [ ]:
chain = A.ProductOfChainsPoset((2,))
constant = A.PModule{OP.QQ}(chain, [1, 1], Dict((1, 2) => ones(OP.QQ, 1, 1)); field=OP.CoreModules.QQField())
tail = OP.generated_submodule(OP.propagate(constant; source=2, vectors=[1]))
@assert OP.dimensions(A.submodule_object(tail)).stalks == [0, 1]
OP.describe(tail)


## What remains unchanged under a basis change?

Exercise: replace the spanning columns for the overlap plane by $(1,1)$ and $(0,1)$. Predict its image dimensions. The actual matrices may change, but the spanned subspace and its image dimensions do not.


In [ ]:
other_columns = OP.propagate(enc; source=6, vectors=[1 0; 1 1])
@assert OP.dimensions(other_columns) == OP.dimensions(plane)
OP.dimensions(other_columns)


For a change of ambient stalk basis, also conjugate the module maps and transform the selected coordinates. Keeping the same numeric vector after changing the ambient basis generally selects a different vector.

For homology modules with retained source cycles, `A.propagation_source_chain(result; target, basis, column=1)` expresses a selected image on the original cells. Supply the actual target-coordinate homology basis. The routine checks the cycles and their independence modulo boundaries; the basis identification with the module is explicitly supplied by you. It cannot reconstruct a basis that was discarded. The resulting `SourceChain` uses the [source-cell views](source_cells.ipynb).

You can also open the existing live module inspector with `OP.inspection_session(N)` to inspect the generated spaces and their maps. Changing the chosen vectors starts a new `propagate` computation.

See the [API contracts](../src/reference/modules.md) for shapes and field semantics, and [the implementation account](../implementation/submodules_and_quotients.md) for how propagation becomes a submodule.
